In [74]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/combined_weather_eda.csv")

df["time"] = pd.to_datetime(df["time"])

df.head()

,time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,weather_code,cloud_cover,wind_speed_10m,wind_direction_10m,latitude,longitude,city,month
0,2025-01-01 00:00:00,18.2,87,18.5,0.0,0.0,0,1,12.0,66,12.9716,77.5946,Bengaluru,1
1,2025-01-01 01:00:00,17.6,88,17.8,0.0,0.0,0,0,11.9,67,12.9716,77.5946,Bengaluru,1
2,2025-01-01 02:00:00,17.1,87,16.9,0.0,0.0,0,2,12.4,64,12.9716,77.5946,Bengaluru,1
3,2025-01-01 03:00:00,16.8,84,16.2,0.0,0.0,0,16,12.8,61,12.9716,77.5946,Bengaluru,1
4,2025-01-01 04:00:00,16.4,83,15.7,0.0,0.0,2,76,12.0,62,12.9716,77.5946,Bengaluru,1


In [75]:
df.shape

(43800, 14)

In [76]:
df["heavy_rain"] = (df["rain"] >= 5).astype(int)

df["heavy_rain"].value_counts()

heavy_rain
0    43556
1      244
Name: count, dtype: int64

In [77]:
df["heavy_rain"].sum()

np.int64(244)

In [78]:
df["hour"] = df["time"].dt.hour
df["month"] = df["time"].dt.month

df["previous_rain"] = df.groupby("city")["rain"].shift(1)

df["previous_rain"] = df["previous_rain"].fillna(0)

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "cloud_cover",
    "wind_speed_10m",
    "previous_rain",
    "month",
    "hour"
]

X = df[features]
y = df["heavy_rain"]

X.head()

,temperature_2m,relative_humidity_2m,cloud_cover,wind_speed_10m,previous_rain,month,hour
0,18.2,87,1,12.0,0.0,1,0
1,17.6,88,0,11.9,0.0,1,1
2,17.1,87,2,12.4,0.0,1,2
3,16.8,84,16,12.8,0.0,1,3
4,16.4,83,76,12.0,0.0,1,4


In [79]:
X.shape, y.shape

((43800, 7), (43800,))

In [80]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (35040, 7)
Testing data: (8760, 7)


In [81]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [82]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9946

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      8711
           1       0.54      0.27      0.36        49

    accuracy                           0.99      8760
   macro avg       0.77      0.63      0.68      8760
weighted avg       0.99      0.99      0.99      8760



In [83]:
y_probability = model.predict_proba(X_test)[:, 1]

y_pred_30 = (y_probability >= 0.30).astype(int)

print(classification_report(y_test, y_pred_30))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      8711
           1       0.30      0.41      0.35        49

    accuracy                           0.99      8760
   macro avg       0.65      0.70      0.67      8760
weighted avg       0.99      0.99      0.99      8760



In [84]:
def predict_heavy_rain(
    temperature,
    humidity,
    cloud_cover,
    wind_speed,
    previous_rain,
    month,
    hour
):
    input_data = pd.DataFrame([{
        "temperature_2m": temperature,
        "relative_humidity_2m": humidity,
        "cloud_cover": cloud_cover,
        "wind_speed_10m": wind_speed,
        "previous_rain": previous_rain,
        "month": month,
        "hour": hour
    }])

    probability = model.predict_proba(input_data)[0][1]

    if probability >= 0.30:
        risk = "High"
    else:
        risk = "Low"

    return {
        "heavy_rain_probability": round(probability, 2),
        "risk": risk
    }

In [ ]:
predict_heavy_rain(
    temperature=50,
    humidity=90,
    cloud_cover=95,
    wind_speed=15,
    previous_rain=4,
    month=7,
    hour=18
)

{'heavy_rain_probability': np.float64(0.0), 'risk': 'Low'}